# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features - this time from a merged cell CSV aready created
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Set your paths here

In [ ]:
# Imports
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# plotting
import plotly.express as px
import plotly.graph_objects as go

%matplotlib inline

import seaborn as sns

from mitolyso_plot_functions import *
from plate_preprocessing import *

from quality_control_functions import *
from single_csv_functions import (
    merge_ij_skeleton_features_into_combined_dataframe_from_folder,
)


## Import the big csv

In [ ]:
# import from a giant csv
# csvpath = "/Volumes/AllieS/Morphology_data/"
csvpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"
filename = "total_combined_cell.csv"

# filename = "total_combined_cell_borders_excluded.csv"
combined_cell_df_mitolyso = pd.read_csv(os.path.join(csvpath, filename))
display(combined_cell_df_mitolyso.shape)

filename_borders_excluded = "total_combined_cell_borders_excluded.csv"

combined_cell_df_mitolyso = fix_column_names(combined_cell_df_mitolyso)


In [ ]:
combined_cell_df_mitolyso["Plate_RowColFieldCode"] = (
    combined_cell_df_mitolyso["Plate_Number"].astype(int).astype(str)
    + "_"
    + combined_cell_df_mitolyso["Metadata_RowColFieldCode"].astype(str)
)


In [ ]:
unique_combinations = combined_cell_df_mitolyso[
    ["Plate_Number", "PassageNumber"]
].drop_duplicates()

combos = list(unique_combinations.itertuples(index=False, name=None))
# display(sorted(combos))
display(combined_cell_df_mitolyso)

In [ ]:
# If using cytodataframe package (https://github.com/cytomining/CytoDataFrame/blob/main/docs/src/examples/cytodataframe_at_a_glance.ipynb)
def display_cytodataframe(
    df,
    original_img_path,
    original_outlines_path=None,
    ch1="MitoTracker_MAX",
    ch2="LAMP1_MAX",
    ch3="DAPI_MAX",
    range=None,
    plate_folder=None,
    mask_folder=None,
    image_code=None,
    image_code_col_name="Plate_RowColFieldCode",
    render_whole_image=False,
    output_path=None,
    transpose=False,
):
    """_summary_

    Args:
        df (Pandas DataFrame): The DataFrame containing the data to display.
        original_img_path (Path or str): the path to the original raw images
        original_outlines_path (Path or str, optional): the path to the CellProfiler image-level outlines. Defaults to None.
        ch1 (str, optional): Channel 1 name. Defaults to "MitoTracker_MAX".
        ch2 (str, optional): Channel 2 name. Defaults to "LAMP1_MAX".
        ch3 (str, optional): Channel 3 name. Defaults to "DAPI_MAX".
        range (list, optional): The range of rows to display. Defaults to [0, 3].
        plate_folder (str, optional): folder for a specific plate. Defaults to None.
        mask_folder (str, optional): folder containing masks for a specific plate. Defaults to None.
        image_code (str, optional): The image code to use. Defaults to None.
        image_code_col_name (str, optional): The column name for the image code. Defaults to "Metadata_Plate_Number_RowColFieldCode".
        render_whole_image (bool, optional): Whether to render the whole image. Defaults to False.
        output_path (str, optional): The path to save the output. Defaults to None.
    """
    from cytodataframe.frame import CytoDataFrame

    df = df.copy()
    # make paths; original_outlines_path is optional
    if isinstance(original_img_path, str):
        original_img_path = Path(original_img_path)
    if isinstance(original_outlines_path, str):
        original_outlines_path = Path(original_outlines_path)

    # if narrowing down to a specific image code, filter the dataframe accordingly
    if image_code is not None:
        df = df[df[image_code_col_name] == image_code]
    if range is None:
        if len(df) > 25:
            print(
                "Warning: Displaying more than 25 rows may be slow, using range=[0,25] by default."
            )
            range = [0, 25]
        else:
            range = [0, len(df)]

    # handle the case where the range is out of bounds for the dataframe
    try:
        df = df.iloc[range[0] : range[1], :]
    except IndexError as e:
        print(f"range {range} not in range of dataframe with {len(df)} rows: {e}")
        print("Displaying up to the last row.")
        df = df.iloc[range[0] : len(df), :]

    # narrow down the dataframe to the specific plate and passage number if provided
    if plate_folder is not None and mask_folder is not None:
        img_path = f"{original_img_path}/{plate_folder}"
        mask_path = f"{original_img_path}/{plate_folder}/{mask_folder}"
        df.loc[
            :,
            [f"Image_PathName_{ch1}", f"Image_PathName_{ch2}", f"Image_PathName_{ch3}"],
        ] = mask_path
    else:
        img_path = f"{original_img_path}"
        mask_path = None

    # set paths
    if original_outlines_path is not None:
        outlines_path = f"{original_outlines_path}"
    else:
        outlines_path = None

    # simplify colnames for display
    df.rename(
        columns={
            "Cell_Unique_ID": "UniqueID",
            "Number_Object_Number": "ObjectNum",
            image_code_col_name: "PlateRowColField",
        },
        inplace=True,
    )
    frame = CytoDataFrame(
        data=df,
        data_context_dir=img_path,
        data_outline_context_dir=outlines_path,
        data_mask_context_dir=mask_path,
        display_options={
            "composite_channels": {
                ch1: "magenta",
                ch2: "yellow",
                ch3: "cyan",
            },
            "equalize_clip_limit": 0.01,
            "brightness": 20,
            # "width": "200px",
            "height": "auto",
            "render_whole_image": render_whole_image,
        },
    )[
        [
            # "ImageNumber",
            "UniqueID",
            "PlateRowColField",
            "ObjectNum",
            f"Image_FileName_{ch1}",
            f"Image_FileName_{ch2}",
            f"Image_FileName_{ch3}",
        ]
    ]
    if transpose:
        display(frame.T)
    else:
        display(frame)
    if output_path is not None:
        frame.to_ome_parquet(output_path)
    return frame


display_cytodataframe(
    combined_cell_df_mitolyso,
    original_img_path="/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/",
    # range=[5, 10],
    original_outlines_path=None,
    ch1="MitoTracker_MAX",
    ch2="LAMP1_MAX",
    ch3="DAPI_MAX",
    image_code_col_name="Plate_RowColFieldCode",
    image_code="6_r05c11f02",
    transpose=False,
)

In [ ]:
intensity_cols = get_intensity_cols(combined_cell_df_mitolyso)
well_avg_cols = [f"{col}_WellAvg" for col in intensity_cols]
norm_intensity_cols = [f"{col}_WellNormalized" for col in intensity_cols]

combined_cell_df_mitolyso_test = normalize_intensity_to_well_average(
    combined_cell_df_mitolyso.copy(), intensity_cols
)
print("Intensity columns:", intensity_cols)
display(
    combined_cell_df_mitolyso_test[
        ["AllGroups", "Plate_Number", "Metadata_Well"]
        + intensity_cols
        + well_avg_cols
        + norm_intensity_cols
    ].head(20)
)

combined_cell_df_mitolyso = normalize_intensity_to_well_average(
    combined_cell_df_mitolyso, intensity_cols
)

## Setup functions


### Remove problematic rows / columns from dataframe

In [ ]:
def remove_problematic_rows_cols(df, plate_number, problematic_rows, problematic_cols, problematic_rows_cols_combo_only=None):
    """
    Remove rows and columns from the dataframe based on the specified plate number.

    Parameters:
    df (pd.DataFrame): The input dataframe.
    plate_number (int): The plate number to filter by.
    problematic_rows (list): List of problematic row numbers to remove.
    problematic_cols (list): List of problematic column numbers to remove.
    problematic_rows_cols_combo_only (list of tuples, optional): List of specific row-column combinations to remove. Each tuple should contain a row number and a column number.

    Returns:
    pd.DataFrame: The filtered dataframe with specified rows and columns removed.
    """
    this_df = df.copy().reset_index(names="Old_Index")  # Create a copy of the dataframe to avoid modifying the original
    # Filter by plate number
    filtered_df = this_df[this_df["Plate_Number"] == plate_number]
    
    problematic_indicies_df = pd.DataFrame()  # Initialize an empty dataframe to store problematic indices
    # if problematic_rows_cols_combo_only:
    # # filter the dataframe to find those combinations
    #     mask = pd.MultiIndex.from_frame(filtered_df[["Metadata_WellRow", "Metadata_WellColumn"]]).isin(problematic_rows_cols_combo_only)
    #     problematic_indicies_df = filtered_df.loc[mask]
    # # Remove problematic rows
    
    # if problematic_rows:
    #     if problematic_indicies_df.empty:
    #         problematic_indicies_df = filtered_df.loc[filtered_df["Metadata_WellRow"].astype(int).isin(problematic_rows)]
    #     else:
    #         problematic_indicies_df = problematic_indicies_df.loc[problematic_indicies_df["Metadata_WellRow"].astype(int).isin(problematic_rows)]

    # # Remove problematic columns
    # if problematic_cols:
    #     if problematic_indicies_df.empty:
    #         problematic_indicies_df = filtered_df.loc[filtered_df["Metadata_WellColumn"].astype(int).isin(problematic_cols)]
    #     else:
    #         problematic_indicies_df = problematic_indicies_df.loc[problematic_indicies_df["Metadata_WellColumn"].astype(int).isin(problematic_cols)]
    
    # # Remove the problematic rows and columns from the original dataframe and preserve the original structure
    if problematic_rows or problematic_cols or problematic_rows_cols_combo_only:
        row_mask = filtered_df["Metadata_WellRow"].astype(int).isin(problematic_rows) if problematic_rows else False
        col_mask = filtered_df["Metadata_WellColumn"].astype(int).isin(problematic_cols) if problematic_cols else False
        combo_mask = (
            pd.MultiIndex.from_frame(filtered_df[["Metadata_WellRow", "Metadata_WellColumn"]])
            .isin(problematic_rows_cols_combo_only)
            if problematic_rows_cols_combo_only
            else False
        )
        problematic_indicies_df = filtered_df.loc[row_mask | col_mask | combo_mask]
    if not problematic_indicies_df.empty:
        print(f"   rows from plate {plate_number} before: {filtered_df.shape}, after: {filtered_df[~filtered_df['Old_Index'].isin(problematic_indicies_df['Old_Index'])].shape}")
        this_df = this_df.loc[~this_df["Old_Index"].isin(problematic_indicies_df["Old_Index"])]
    else:
        print(f"No problematic rows or columns found for plate {plate_number}. No rows removed.")
    return this_df.drop(columns=["Old_Index"]).reset_index(drop=True)  # Drop the temporary index column before returning

plate_numbers_problematic_rows_cols = {
    1: {"problematic_rows": [2, 7], "problematic_cols": [4], "problematic_rows_cols_combo_only": None},
    2: {"problematic_rows": [], "problematic_cols": [], "problematic_rows_cols_combo_only": [(2,4),(3,4)]},
    3: {"problematic_rows": [2], "problematic_cols": [], "problematic_rows_cols_combo_only": [(3,4),(4,4)]},
    4: {"problematic_rows": [2], "problematic_cols": [], "problematic_rows_cols_combo_only": [(3,4),(4,4)]},
    5: {"problematic_rows": [], "problematic_cols": [], "problematic_rows_cols_combo_only": [(4,4),(5,4)]},
    6: {"problematic_rows": [], "problematic_cols": [], "problematic_rows_cols_combo_only": [(2,2),(3,2)]},
    7: {"problematic_rows": [], "problematic_cols": [], "problematic_rows_cols_combo_only": None},
}

filter_rows_df = combined_cell_df_mitolyso.copy()  # Start with the original dataframe
for plate_number in plate_numbers_problematic_rows_cols:
    problematic_rows = plate_numbers_problematic_rows_cols[plate_number]["problematic_rows"]
    problematic_cols = plate_numbers_problematic_rows_cols[plate_number]["problematic_cols"]
    problematic_rows_cols_combo_only = plate_numbers_problematic_rows_cols[plate_number]["problematic_rows_cols_combo_only"]
    if problematic_rows or problematic_cols or problematic_rows_cols_combo_only:
        print(f"Removing problematic rows and/or columns for plate {plate_number}:")
        if problematic_rows:
            print(f"  Problematic rows: {problematic_rows}")
        if problematic_cols:
            print(f"  Problematic columns: {problematic_cols}")
        if problematic_rows_cols_combo_only:
            print(f"  Problematic row-column combinations: {problematic_rows_cols_combo_only}")
        shape_before =filter_rows_df.shape
        filter_rows_df = remove_problematic_rows_cols(
            filter_rows_df, plate_number=plate_number, problematic_rows=problematic_rows, problematic_cols=problematic_cols, problematic_rows_cols_combo_only=problematic_rows_cols_combo_only
        )
        print(f"  Shape of full df before: {shape_before}, shape after: {filter_rows_df.shape}")
    else:
        print(f"No problematic rows or columns specified for plate {plate_number}. No rows removed.")

print(f"Original dataframe before: {combined_cell_df_mitolyso.shape}, shape after: {filter_rows_df.shape}")


### Filter out the poorly segmented cells based on the previously defined quality control filters and rename the columns

In [ ]:
# filtered_combined_cell_df_mitolyso, filtered_combined_summary_df = (
#     get_filtered_df_and_export_summary_to_csv(
#             combined_cell_df_mitolyso,
#             group_col="Plate_Number",
#             groups=sorted(combined_cell_df_mitolyso["Plate_Number"].unique()),
#             savepath="filtering_summaries",
#             individual_filters=False,
#         )
# )
# filtered_combined_cell_df_mitolyso, filtered_combined_summary_df = (
#     get_filtered_df_and_export_summary_to_csv(
#         combined_cell_df_mitolyso,
#         group_col="AllGroups",
#         groups=get_all_group_order(),
#         savepath="filtering_summaries",
#         individual_filters=False
#     )
# )

In [ ]:
final_filtered_df, final_filtered_summary_df = get_filtered_df_and_export_summary_to_csv(
    filtered_rows_df,
    reference_df=filtered_rows_df,
    savepath="filtering_summaries",
)

## Search Column Names

In [ ]:
# colnames
# sns.barplot(filter_df_2, x="AllGroups",y="AreaShape_Area", hue="Plate_Number", palette=colour_dict)
colnames = search_column_name(combined_cell_df_mitolyso, "Plate")
display(combined_cell_df_mitolyso[colnames])

In [ ]:
test_query_df = combined_cell_df_mitolyso.query(
    "Plate_Number == 6 and Metadata_WellColumn == 2"
)

# display(test_query_df[[
#     "Plate_Number",
#     "Metadata_RowColFieldCode",
#     "Number_Object_Number",
#     "AllGroups",
#     "AreaShape_Area",
#     "Children_Mitochondria_Count",
#     "Children_Lysosomes_Count",
#     "Intensity_MeanIntensity_LAMP1_MAX",
#     "Intensity_MeanIntensity_MitoTracker_MAX",
# ]].head(20))
# display(
#     combined_cell_df_mitolyso.groupby(["Plate_Number", "Metadata_WellRow"])[
#         "Intensity_MeanIntensity_LAMP1_MAX"
#     ].describe()
# )


In [ ]:
def compare_features_barplots(
    df,
    feature1,
    feature2,
    group_col="Plate_Number",
    hue_col="Metadata_WellColumn",
    figsize=(12, 8),
):
    fig, ax = plt.subplots(2, 1, figsize=figsize, sharex=True)
    sns.barplot(
        data=df,
        ax=ax[0],
        x=group_col,
        y=feature1,
        hue=hue_col,
        palette="tab10",
        legend=True,
    )
    sns.barplot(
        data=df,
        ax=ax[1],
        x=group_col,
        y=feature2,
        hue=hue_col,
        palette="tab10",
        legend=False,
    )
    ax[0].legend(
        loc="upper right", bbox_to_anchor=(1.18, 1), borderaxespad=0, title=hue_col
    )
    plt.show()


compare_features_barplots(
    final_filtered_df,
    "Intensity_MeanIntensity_LAMP1_MAX",
    "AreaShape_Area",
    hue_col="Plate_Number",
    group_col="Metadata_WellRow",
    figsize=(12, 8)
)
# fig,ax =plt.subplots(2, 1,figsize=(12, 6),sharex=True)
# sns.barplot(combined_cell_df_mitolyso, ax=ax[0], hue="Metadata_WellColumn", y="Children_Lysosomes_Count", x="Plate_Number", palette="tab10",legend=True)
# sns.barplot(combined_cell_df_mitolyso, ax=ax[1], hue="Metadata_WellColumn", y="Children_Mitochondria_Count", x="Plate_Number", palette="tab10",legend=False)
# ax[0].legend(loc="upper right", bbox_to_anchor=(1.1, 1), borderaxespad=0, title="Well Column")
# plt.show()


# Define the cell features


### Join in imageJ skeleton features
Side note; make sure to use TotalMitochondria_Object_Number for the skeleton analysis if I want to join the correct items

In [ ]:
# Add ijskeleton columns
ij_csvpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/IJ_Output/"
final_filtered_df["Metadata_PlateNumber"] = final_filtered_df["Plate_Number"].astype(
    int
)

final_filtered_df_with_ijskeleton = (
    merge_ij_skeleton_features_into_combined_dataframe_from_folder(
        final_filtered_df,
        ij_csvpath,
        ij_keys=["Metadata_PlateNumber", "Metadata_RowColField", "ObjectNumber"],
        parent_keys=[
            "Metadata_PlateNumber",
            "Metadata_RowColFieldCode",
            "TotalMitochondria_Number_Object_Number",
        ],
        show=True,
        check_duplicates=True,
    )
)

def fix_ij_skeleton_column_names(df, colnames=None):
    """Fix column names for IJ skeleton features in the dataframe.

    Args:
        df (pd.DataFrame): The DataFrame to fix column names for.
        colnames (list of str, optional): optional override for column names to fix. Defaults to None.

    Returns:
        pd.DataFrame: The DataFrame with fixed column names.
    """    
    df = df.copy()
    if colnames is None:
        colnames = search_column_name(df, "IJ_Mitochondria")
    new_colnames = {}   
    for col in colnames:
#         - Rename some features to be more intuitive
#   - IJ_Mitochondria_Masks_BranchLength_MaxAcrossAllStructures and IJ_Mitochondria_Masks_LargestStructure_MaximumBranchLength are the same, drop the former so its consistent
#   - same with IJ_Mitochondria_Masks_BranchesPerStructure_MaxAcrossAllStructures and IJ_Mitochondria_Masks_LargestStructure_NumberOfBranches
#   - "IJ_Mitochondria_Masks_SkeletonLength_MaxAcrossAllStructures" should be renamed to "IJ_Mitochondria_Masks_LargestStructure_SkeletonLength"
#   - "IJ_Mitochondria_Masks_TotalAcrossAllStructures_SkeletonLength" and "IJ_Mitochondria_Masks_TotalAcrossAllStructures_SkeletonLength_FromBranches" are basically the exact same, remove the latter for simplicity
    
        #Make _Max_ more clear for largest structure
        if "_Max_" in col:
            #for largest structure
            new_colname = col.replace("_Max_", "_LargestStructure_")
        elif "_Total_" in col:
            #for total across all structures
            new_colname = col.replace("_Total_", "_TotalAcrossAllStructures_")
        elif col.endswith("_Max"):
            #for the ambigous _Max at the end of the column name
            if "SkeletonLength" in col:
                new_colname = col.replace("_SkeletonLength_Max", "_LargestStructure_SkeletonLength")
            else:
                new_colname = col.replace("_Max", "_toRemove")
        else:
            new_colname = col
        new_colnames[col] = new_colname
    df.rename(columns=new_colnames, inplace=True)
    df.drop(columns=[col for col in new_colnames.values() if col.endswith(("_toRemove", "_FromBranches"))], inplace=True, errors="ignore")
    return df

final_filtered_df_with_ijskeleton = fix_ij_skeleton_column_names(final_filtered_df_with_ijskeleton)

display(
    final_filtered_df_with_ijskeleton[
        final_filtered_df_with_ijskeleton["Metadata_PlateNumber"] == 4
    ][
        [
            "Metadata_PlateNumber",
            "Metadata_RowColFieldCode",
            "TotalMitochondria_Number_Object_Number",
            "TotalMitochondria_AreaShape_Area",
            "IJ_Mitochondria_Masks_MitochondrialFootprint"
        ]
        + search_column_name(final_filtered_df_with_ijskeleton, "IJ_Mitochondria")
    ].head(20)
)

In [ ]:
# csv_output_path = Path("/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs")
# os.makedirs(csv_output_path, exist_ok=True)
# final_filtered_df_with_ijskeleton.to_csv(csv_output_path / "total_combined_cell_filtered_ij.csv", index=False)

In [ ]:
# WIP Add extra columns
colour_dict = get_hard_code_plate_colours(final_filtered_df_with_ijskeleton)


def get_unique_cols_to_use(df):
    base_cols = [
        # "FileName_MitoTracker_MAX",
        "Metadata_PlateNumber",
        "Metadata_RowColFieldCode",
        "AllGroups",
        "AreaShape_Area",
    ]
    areashape_features = [
        "AreaShape_Area",
        "AreaShape_Perimeter",
        "AreaShape_EquivalentDiameter",
        "AreaShape_Eccentricity",
        "AreaShape_FormFactor",
        "AreaShape_Solidity",
        "AreaShape_Extent",
        "AreaShape_MaxFeretDiameter",
        "AreaShape_MinFeretDiameter",
        "AreaShape_MeanRadius",
    ]

    colnames_mitoskel_nuc = search_column_name(df, "Nuclei_ObjectSkeleton")
    colnames_mitocount = search_column_name(
        df, ["Children_Mitochondria", "Count"], inclusive_or=False
    )
    colnames_mitoarea = search_column_name(
        df, ["Mito", "AreaShape_Area"], inclusive_or=False
    )

    colnames_lysocount = search_column_name(
        df, ["Children_Lysosomes", "Count"], inclusive_or=False
    )
    colnames_lysoarea = search_column_name(df, areashape_features, inclusive_or=True)
    for col in colnames_lysoarea.copy():
        if "Lyso" in col and "AreaShape" in col:
            continue
        else:
            colnames_lysoarea.remove(col)

    colnames_intesnity_distribution = search_column_name(
        df, ["Radial", "MitoTracker"], inclusive_or=False
    )
    colnames_intesnity_distribution += search_column_name(
        df, ["Radial", "LAMP1"], inclusive_or=False
    )
    for col in colnames_intesnity_distribution.copy():
        if "Nuclei" in col or "Closing" in col:
            colnames_intesnity_distribution.remove(col)
        else:
            continue

    colnames_overlap = search_column_name(
        df, ["Overlap", "Correlation"], inclusive_or=True
    )
    for col in colnames_overlap.copy():
        if ("Mito" not in col and "Lyso" not in col and "LAMP1" not in col) or (
            "Texture" in col or "DAPI" in col
        ):
            colnames_overlap.remove(col)
        else:
            continue
    print(colnames_overlap)

    colnames_ij = search_column_name(df, "IJ_Mitochondria")

    use_cols = (
        base_cols
        + colnames_mitoskel_nuc
        + colnames_ij
        + colnames_mitocount
        + colnames_mitoarea
        + colnames_lysocount
        + colnames_lysoarea
        + colnames_overlap
        + colnames_intesnity_distribution
    )
    use_cols_unique = list(dict.fromkeys(use_cols))
    # scrub out any non-numeric columns that we aren't going to use for analysis
    use_cols_unique_copy = use_cols_unique.copy()
    for col in use_cols_unique_copy:
        if col in base_cols:
            continue
        elif "Metadata" in col or "Title" in col or "FileName" in col:
            use_cols_unique.remove(col)
    return use_cols_unique


def get_object_skeleton_length_cols(df):
    colnames_object_skeleton_length = search_column_name(df, "SkeletonLength")
    for col in colnames_object_skeleton_length.copy():
        if (
            "Mean" in col
            or "Median" in col
            or "Threshold" in col
            or "Stdev" in col
            or "FromBranches" in col
        ):
            colnames_object_skeleton_length.remove(col)

    print(colnames_object_skeleton_length)
    return colnames_object_skeleton_length


skeleton_length_cols = get_object_skeleton_length_cols(
    final_filtered_df_with_ijskeleton
)
print(skeleton_length_cols)
cols_to_use = get_unique_cols_to_use(final_filtered_df_with_ijskeleton)


In [ ]:
def make_per_cell_area_column_names(
    df,
    use_cols,
    area_col="AreaShape_Area",
    colnames_mitoskel_seeds=None,
    number_of_seeds_col="Children_MitoSkel_Seeds_Count",
    areashape_cols=None,
    calculate_totals=False,
    base_cols=None,
    exclude_per_area=None,
):
    if base_cols is None:
        base_cols = [
            "Metadata_PlateNumber",
            "Metadata_RowColField",
            "AllGroups",
            "AreaShape_Area",
        ]
    if areashape_cols is None:
        areashape_cols = [
            "AreaShape_Area",
            "AreaShape_Perimeter",
            "AreaShape_EquivalentDiameter",
        ]
    df = df.copy()
    new_use_cols = use_cols.copy()
    for col in base_cols:
        if col in new_use_cols:
            new_use_cols.remove(col)

    new_columns = {}

    count_flag = 0
    for col in areashape_cols:
        # remove the col from new_use_cols so that it doesn't get processed again in the final loop
        if col in new_use_cols:
            new_use_cols.remove(col)

        df[col] = pd.to_numeric(df[col], errors="coerce")

        if calculate_totals and "Mean" in col:
            # calculate the total area occupied and then divide by area
            col_without_mean = col.replace("Mean_", "")
            new_columns["Math_Total_" + col_without_mean] = (
                df[col] * df[count_cols[count_flag]]
            )
            new_columns["Per_Area_AreaOccupied_" + col_without_mean] = (
                df["Math_Total_" + col_without_mean] / df[area_col]
            )

        elif "Mean" in col:
            continue
        elif "Total" in col:
            col_without_total = col.replace("Total_", "")
            new_columns["Per_Area_AreaOccupied_" + col_without_total] = (
                df[col] / df[area_col]
            )
        elif "RelabeledMito" in col:
            new_columns["Per_Area_AreaOccupied_" + col] = df[col] / df[area_col]
        else:
            new_columns["Per_Area_" + col] = df[col] / df[area_col]

    # calculate total per cell for mito skel seeded version (if using), then divide by area
    if colnames_mitoskel_seeds:
        for col in colnames_mitoskel_seeds:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            col_without_mean = col.replace("Mean_", "")
            new_columns["Math_Total_" + col_without_mean] = (
                df[col] * df[number_of_seeds_col]
            )
            new_columns["Per_Area_" + col_without_mean] = (
                new_columns["Math_Total_" + col_without_mean] / df[area_col]
            )
            if col in new_use_cols:
                new_use_cols.remove(col)

    # for everything else in use_cols that makes sense, just divide by area
    if exclude_per_area is None:
        exclude_per_area = [
            "Mean",
            "Median",
            "Std",
            "Distribution",
            "Metadata",
            "Title",
            "Per_Area",
            "Location",
            "Correlation",
            "Overlap",
            "Eccentricity",
            "FormFactor",
            "Solidity",
            "Extent",
            "BranchLength",
        ]
    for col in new_use_cols:
        if any(exclude in col for exclude in exclude_per_area):
            continue
        else:
            print("making per area column for:", col)
            df[col] = pd.to_numeric(df[col], errors="coerce")
            # make new colname by adding per area
            new_colname = "Per_Area_" + col
            if "Count" in col:
                # make new colname by replacing Children_ with Number_ for readbility
                new_colname = new_colname.replace("Children_", "Number_")

            new_columns[new_colname] = df[col] / df[area_col]

    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    df = pd.concat([df, new_columns_df], axis=1)
    return df


def make_per_skeleton_length_column_names(
    df,
    use_cols,
    skeleton_length_cols,
    base_cols=None,
    colnames_mitoskel_seeds=None,
    sets=None,
    feature_types=None,
):
    if sets is None:
        sets = []
    if feature_types is None:
        feature_types = [
            "Nuclei_ObjectSkeleton",
            "MitoSkel_Seeds_ObjectSkeleton",
            "IJ_Mitochondria",
        ]
    df = df.copy()
    new_use_cols = use_cols.copy()
    if base_cols is None:
        base_cols = [
            "Metadata_PlateNumber",
            "Metadata_RowColField",
            "AllGroups",
            "AreaShape_Area",
        ]
    # take out things that don't make sense
    for col in use_cols:
        if (
            col in base_cols
            or col in skeleton_length_cols
            or "Metadata" in col
            or "Title" in col
            or "Mean" in col
            or "Median" in col
            or "Stdev" in col
            or "Footprint" in col
            or "Per_Area" in col
            or "Correlation" in col
            or "Overlap" in col
            or "Distribution" in col
            or "AreaShape" in col
            or "Children" in col
        ):
            new_use_cols.remove(col)

    new_columns = {}

    for col in new_use_cols:
        # check if the column is one of the feature types we want to process
        this_feature_type = None
        for feature_type in feature_types:
            if feature_type in col:
                this_feature_type = feature_type
                break

        df[col] = pd.to_numeric(df[col], errors="coerce")

        for length_col in skeleton_length_cols:
            # only divide by the skeleton length column that matches the feature type of the current column
            if this_feature_type and this_feature_type not in length_col:
                continue
            elif this_feature_type == "IJ_Mitochondria":
                subtypes = ["_LargestStructure", "_TotalAcrossAllStructures"]
                if any(subtype in col for subtype in subtypes) and any(
                    (subtype in col and subtype not in length_col)
                    or (subtype not in col and subtype in length_col)
                    for subtype in subtypes
                ):
                    continue
            print(f"Calculating Per_SkeletonLength for {col} using {length_col}")
            if sets and any(set_name in length_col for set_name in sets):
                for set_name in sets:
                    if set_name in length_col and set_name in col:
                        print(f"Using set {set_name} for {col} and {length_col}")
                        new_columns["Per_SkeletonLength_" + col] = (
                            df[col] / df[length_col]
                        )
                        break
            else:
                new_columns["Per_SkeletonLength_" + col] = df[col] / df[length_col]

    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    df = pd.concat([df, new_columns_df], axis=1)
    return df


extrafeatures_filtered_cell_df_mitolyso = make_per_cell_area_column_names(
    final_filtered_df_with_ijskeleton,
    use_cols=cols_to_use,
)
display(extrafeatures_filtered_cell_df_mitolyso.head())
extrafeatures_filtered_cell_df_mitolyso = make_per_skeleton_length_column_names(
    extrafeatures_filtered_cell_df_mitolyso,
    use_cols=cols_to_use,
    skeleton_length_cols=skeleton_length_cols,
    feature_types=[
        "Nuclei_ObjectSkeleton",
        "MitoSkel_Seeds_ObjectSkeleton",
        "IJ_Mitochondria",
    ],
)


In [ ]:


# Calculate the total area occupied by mitochondria and lysosomes per cell
def calculate_extra_features(full_df, organelles=["Mitochondria", "Lysosomes"]):
    """_summary_

    Args:
        full_df (DataFrame): _description_

    Returns:
        df (DataFrame): the df with all the feature calcs
    """
    df = full_df.copy()
    # Total intensity per cell based on integrated instensity if I don't already have the merged area
    for organelle in organelles:
        if organelle == "Mitochondria":
            tag = "MitoTracker"
            name = "TotalMitochondria"
            math = None
        elif organelle == "Lysosomes":
            tag = "LAMP1"
            name = "TotalLysosomes"
            math = None
        else:
            continue
        df[f"Mean_Intensity_Per_{organelle}_PerCell_Area"] = (
            mean_intensity_per_compartment_per_cell(df, organelle, name, tag, math=math)
        )
        # Compartment diameter ratios
        df[f"Ratio_Mean_{organelle}_MaxMinFeret_DiameterRatio"] = (
            df[f"Mean_{organelle}_AreaShape_MaxFeretDiameter"]
            / df[f"Mean_{organelle}_AreaShape_MinFeretDiameter"]
        )
        # organelle is also prefix here
        df[f"Ratio_Median_{organelle}_DiameterRatio_PerCell"] = (
            df[f"{organelle}_Median_{organelle}_AreaShape_MaxFeretDiameter"]
            / df[f"{organelle}_Median_{organelle}_AreaShape_MinFeretDiameter"]
        )

        # Quin's Ratio: Ratio of centroid distance to minimum distance for mitochondria and lysosomes
        # increase = more peripheral, decrease = more nuclear
        df[f"Ratio_Mean_{organelle}_Distance_Centroid_Cell_Minimum_Cell_QuinRatio"] = (
            df[f"Mean_{organelle}_Distance_Centroid_Cell"]
            / df[f"Mean_{organelle}_Distance_Minimum_Cell"]
        )

        # Distance to parents percellarea
        df[f"Per_Area_Mean_{organelle}_Distance_Centroid_Cell"] = (
            df[f"Mean_{organelle}_Distance_Centroid_Cell"] / df["AreaShape_Area"]
        )
        df[f"Per_Area_Mean_{organelle}_Distance_Minimum_Cell"] = (
            df[f"Mean_{organelle}_Distance_Minimum_Cell"] / df["AreaShape_Area"]
        )

    # mitolyso related
    df["Ratio_Number_Lysosomes_To_Mitochondria"] = (
        df["Children_Lysosomes_Count"] / df["Children_Mitochondria_Count"]
    )
    # df["Density_Lysosomes_Mitochondria_Ratio"] = (
    #     df["OccupiedAreaFraction_Lysosomes_PerCell_Area"]
    #     / df["OccupiedAreaFraction_Mitochondria_PerCell_Area"]
    # )
    df["Ratio_Area_Lysosomes_To_Mitochondria"] = (
        df["TotalLysosomes_AreaShape_Area"] / df["TotalMitochondria_AreaShape_Area"]
    )

    return df


# display(extrafeatures_filtered_cell_df_mitolyso)


### Make a convenient table for display and/or plots

In [ ]:
def make_display_df_and_pivot_tables(df, use_cols=None, make_new_features=False, display_table=True):
    if use_cols is None:
        use_cols = get_unique_cols_to_use(df)
    if make_new_features:
        df = calculate_extra_features(df)
        df = make_per_cell_area_column_names(
            df, use_cols=use_cols
        )
        skeleton_length_cols = get_object_skeleton_length_cols(df)
        df = make_per_skeleton_length_column_names(
            df,
            use_cols=use_cols,
            skeleton_length_cols=skeleton_length_cols,
            feature_types=[
                "Nuclei_ObjectSkeleton",
                "MitoSkel_Seeds_ObjectSkeleton",
                "IJ_Mitochondria",
            ],
        )
    colnames_per_area = search_column_name(df, "Per_Area")
    colnames_per_skeletonlength = search_column_name(df, "Per_SkeletonLength")
    use_cols_new = use_cols + colnames_per_area + colnames_per_skeletonlength
    use_cols_new_unique =list(dict.fromkeys(use_cols_new))
    
  
    display_df = df[use_cols_new_unique]
    display_df_pivot_plates = display_df.pivot_table(
        index=["Metadata_PlateNumber", "AllGroups"],
        values=use_cols_new_unique[3:],
        #columns=["AllGroups"],
        aggfunc="mean",
    )
    if display_table:
        display(display_df)
        display(display_df_pivot_plates)
    return display_df, display_df_pivot_plates

display_df, display_df_pivot_plates = make_display_df_and_pivot_tables(
    extrafeatures_filtered_cell_df_mitolyso,
    use_cols=cols_to_use,
    make_new_features=False,
    display_table=True
    )



### Make feature dictionaries

In [ ]:
def get_feature_dicts(df):
    columns_list = define_cell_features(df)
    mito_features = make_feature_dict(
        [
            col
            for col in columns_list
            if ("Mito" in col or "Mitochondria" in col)
            and ("DAPI" not in col and "LAMP1" not in col and "Frame" not in col)
            and not (col.startswith("Nuclei_"))
        ]
    )
    lyso_features = make_feature_dict(
        [
            col
            for col in columns_list
            if ("Lysosome" in col or "LAMP1" in col or "Lyso" in col)
            and ("DAPI" not in col and "Mito" not in col and "Frame" not in col)
            and not (col.startswith("Nuclei_"))
        ]
    )
    nuc_features = make_feature_dict(
        [
            col
            for col in columns_list
            if ("Nuc" in col or "DAPI" in col)
            and ("MitoTracker" not in col and "LAMP1" not in col and "Frame" not in col)
        ]
    )
    cell_features = make_feature_dict(
        [
            col
            for col in columns_list
            if "AreaShape" in col
            and "Mito" not in col
            and "Lyso" not in col
            and "LAMP1" not in col
            and "Nuc" not in col
            and "DAPI" not in col
            and "Metadata" not in col
            and "FileName" not in col
            and "PathName" not in col
        ]
    )
    return [mito_features, lyso_features, nuc_features, cell_features]

## Feature lists here:

In [ ]:
feature_dicts = get_feature_dicts(extrafeatures_filtered_cell_df_mitolyso)
feature_names = [
    "Mitochondria Features",
    "Lysosome Features",
    "Nucleus Features",
    "Cell Features",
]

# Define the output file path
output_file_path = "allfeatures_file.md"

# Open the file in write mode
with open(output_file_path, "w") as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f'"{feature}",\n')
            file.write("\n")

print(f"List has been written to {output_file_path}")


In [ ]:
test_df = extrafeatures_filtered_cell_df_mitolyso
pivot_df = test_df.pivot_table(
    index=["Metadata_PlateNumber", "AllGroups"],
    values=["AreaShape_Area"],
    aggfunc="mean",
)
display(pivot_df)

# Functions for Data Analysis
calulcate normalizations, remove extreme left outliers, etc

## Normalize features to control (Passage 6-8)

In [ ]:
# updated/faster version that doesn't use apply and is much faster for large dataframes
valid_feature_cols = get_valid_numeric_features(
    extrafeatures_filtered_cell_df_mitolyso, feature_dicts
)

group_col = "AgeGroup"
control_value = 0

norm_combined_cell_df_mitolyso = normalize_quantities_to_control_group_average(
    extrafeatures_filtered_cell_df_mitolyso.copy(),
    valid_feature_cols,
    group_col,
    control_value,
    overwrite=True,
    drop_avg_cols=True,
)


# It's plotting time


## Most interesting features so far
- Intensity_MassDisplacment_MitoTracker - increase
- Median Mitochondria Location CenterMass Intensity X
  - decrease,splits into bimodal
  - similar for y
  - AreaShape_Center_X and Center_Y also have similar pattern
  - And Location_Center
- Median mitochondria loaction max intensity
  - Same as CenterMass
- Mean Mitochondria Solidity - increase to p29
- Children Mitochondria Count - gradual increase
  - But scales with size - Density is not sig (slight increase)
  - Total mito area increases; complements this
- Mean Mito Centroid distance - increased (more peripheral)
  - But median is much more vairable between batches
- Mean Mito Trunks - decreased
- Median mito area per cell area - increased
### Lysosomes
- Total area also goes up
- rep3 looks like an outlier here
- Also has the location_ maxintensity change and go bimodal
- bimodal-looking intensity
### Nuclei
- Nuc area increases; scales with cell size increases
- solidity, extent down
- Nuc area ratio - slight increase


## Distribution plots

In [ ]:
# Functions to visualize the distributions
# Define and use a simple function to label the plot in axes coordinates
def ridge_label(x, color, label):
    ax = plt.gca()
    ax.text(
        -0.1,
        -0.2,
        label,
        fontweight="bold",
        color=color,
        ha="left",
        va="center",
        transform=ax.transAxes,
    )


def seaborn_ridgeplot(
    df,
    value_col,
    group_col,
    palette=None,
    bw_adjust=1,
    xlabel=None,
    xlim=(None, None),
    title=None,
    fill_alpha=1,
    linewidth=1.5,
    figsize=(20, 30),
    save=True,
    out_dir="",
    show_percentiles=True,
    truncate_outliers=True,
    norm=False,
):
    """
    Make a ridgeline (joyplot) using seaborn FacetGrid and kdeplot
    Args:
        df: DataFrame
        value_col: str, column with numeric values
        group_col: str, column with group/category
        palette: seaborn palette or list/dict of colors
        bw_adjust: float, KDE bandwidth adjust
        xlabel: str or None
        title: str or None
        fill_alpha: float, alpha for fill
        linewidth: float, line width for outline
        figsize: tuple, figure size
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

    unique_groups = df[group_col].unique()
    if palette is None:
        palette = sns.cubehelix_palette(len(unique_groups), rot=-0.25, light=0.7)
    else:
        palette = palette

    if truncate_outliers and xlim == (None, None):
        try:
            if norm:
                top_fence = df[value_col].mean() + 10 * df[value_col].std()
                bottom_fence = None  # np.percentile(data_df[y_value], 0.000001)
            else:
                top_fence = np.percentile(df[value_col], 99.9)
                bottom_fence = np.percentile(df[value_col], 0.01)
                # handle errors where the data is very skewed and the percentile is inf or nan
                if top_fence == 0 or np.isnan(top_fence) or np.isinf(top_fence):
                    top_fence = None
                if (
                    bottom_fence == 0
                    or np.isnan(bottom_fence)
                    or np.isinf(bottom_fence)
                ):
                    bottom_fence = None
            xlim = (bottom_fence, top_fence)
        except ValueError as e:
            print(e)
            top_fence = None
            bottom_fence = None
            xlim = (None, None)
        print(f"Truncating outliers at: {xlim}")
        xmin, xmax = xlim
        plot_df = df.copy()
        if xmin is not None:
            plot_df = plot_df[plot_df[value_col] >= xmin]
        if xmax is not None:
            plot_df = plot_df[plot_df[value_col] <= xmax]
    else:
        plot_df = df.copy()
    # Initialize the FacetGrid object
    g = sns.FacetGrid(
        plot_df,
        row=group_col,
        hue=group_col,
        aspect=15,
        height=0.5,
        palette=palette,
        xlim=xlim,
    )
    # Draw the densities in a few steps
    g.map(
        sns.kdeplot,
        value_col,
        bw_adjust=bw_adjust,
        clip_on=False,
        fill=True,
        alpha=fill_alpha,
        linewidth=linewidth,
    )
    g.map(sns.kdeplot, value_col, clip_on=False, color="w", lw=2, bw_adjust=bw_adjust)

    if show_percentiles:
        percentiles = [5, 12.5, 25, 50, 75, 87.5, 95]
        for ax in g.axes.flatten():
            # group label text (FacetGrid puts "group_col = <value>" in the title)
            title_text = ax.get_title()
            if " = " in title_text:
                group_val = title_text.split(" = ", 1)[1]
            else:
                group_val = title_text

            group_data = df[df[group_col] == group_val][value_col].dropna()
            if group_data.empty:
                continue

            # pick the most representative line on the axis (the KDE line)
            lines = ax.get_lines()
            if not lines:
                continue
            # choose the line with the largest x-range (robust against multiple lines)
            kde_line = max(lines, key=lambda l: np.ptp(l.get_xdata()))
            xs = kde_line.get_xdata()
            ys = kde_line.get_ydata()
            # line_colour = kde_line.get
            # compute median and interpolate its KDE height
            median = np.median(group_data)
            height = np.interp(median, xs, ys, left=0.0, right=0.0)

            # draw a solid thicker median line from y=0 up to the KDE height
            ax.vlines(
                median, 0, height, color="black", linewidth=3, linestyle=":", zorder=4
            )
            ax.set_xlim(xlim)
            # draw lighter dashed lines for the percentiles (optional)
            group_percentiles = np.percentile(group_data, percentiles)
            for p in group_percentiles:
                ax.vlines(p, 0, height, color="dimgray", ls=":", alpha=0.6, linewidth=2)
    # passing color=None to refline() uses the hue mapping
    g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)

    g.map(ridge_label, value_col)

    # Set the subplots to overlap
    g.figure.subplots_adjust(hspace=-0.25)
    g.figure.set_size_inches(figsize)
    # Remove axes details that don't play well with overlap
    g.set_titles("")
    g.set(yticks=[], ylabel="", xlim=xlim)
    g.despine(bottom=True, left=True)
    if xlabel:
        plt.xlabel(xlabel, fontweight="bold", fontsize=14)
    else:
        plt.xlabel(value_col, fontweight="bold", fontsize=14)
    if title:
        g.figure.suptitle(title, ha="right", fontsize=18, fontweight="bold")
    plt.xlim(xlim)
    plt.tight_layout()
    if save:
        plt.savefig(f"{Path(out_dir, f'{value_col}_{group_col}_joyplot')}.png")
    plt.show()



## Helper functions for plot building

In [ ]:
from mitolyso_plot_functions import (
    super_splitviolinplot_helper_singleplot 
    single_feature_super_splitviolinplot 
    super_boxplot_helper_singleplot
    single_feature_super_boxplot
)

## Make a plot for a single feature

In [ ]:
order = get_all_group_order()
feature_meas = "AreaShape_Area"  # "AreaShape_Area"#Mean_Lysosomes_Distance_Centroid_Nuclei_PerCell_Area" #Mean_Lysosomes_DiameterRatio_PerCell"
ylabel = None  # "Mitochondrial Density Per Cell (relative to youngest passage)"#None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"

pairs = getpairs(combined_cell_df_mitolyso, group, order)

data_df = extrafeatures_filtered_cell_df_mitolyso.copy()
#this_df = norm_combined_cell_df_mitolyso.copy()

# Map each plate to a color and hard code that shit
colour_dict = get_hard_code_plate_colours(data_df)
pallete = colour_dict  # "pastel"

remove_outliers = False
reps_to_exclude = []
plot_dir = "plots/notnorm"
os.makedirs(plot_dir, exist_ok=True)

figsize = (8, 8)

single_feature_super_splitviolinplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    plate_col_name="Plate_Number",
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=False,
    order=order,
    test="tukey_v3",
    reps_to_exclude=reps_to_exclude,
    show_hist=False,  # True,
    remove_outliers=remove_outliers,
    rm_outliers_method="gesd_2",
    truncate_outliers=True,
    truncate_outlier_percentile=97.5,
    legend=True,
    context="talk",
    font_scale=0.8,
    figsize=figsize,
    pallete=pallete,
    p_correction="fdr_bh",
    annotation_location="inside",
    # truncate_outliers=True
)

# NOTE: R5 has smallest cells in p23-25, which is also highest mito density. Also note plate 3 has lysosome staining abnormalities in the first two cols

### Using SuperBoxPlots

In [ ]:
order = get_all_group_order()
feature_meas = "AreaShape_Area"  # "AreaShape_Area"#Mean_Lysosomes_Distance_Centroid_Nuclei_PerCell_Area" #Mean_Lysosomes_DiameterRatio_PerCell"
ylabel = None  # "Mitochondrial Density Per Cell (relative to youngest passage)"#None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"

pairs = getpairs(final_filtered_df_with_ijskeleton, group, order)

data_df = extrafeatures_filtered_cell_df_mitolyso.copy()

# Map each plate to a color and hard code that shit
colour_dict = get_hard_code_plate_colours(data_df)
pallete = colour_dict  # "pastel"

plot_dir = "plots/notnorm"
os.makedirs(plot_dir, exist_ok=True)

figsize = (12, 10)

single_feature_super_boxplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    plate_col_name="Plate_Number",
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=False,
    order=order,
    test="tukey_v3",
    truncate_outliers=False,
    limit_whiskers=True,
    legend=False,
    context="poster",
    font_scale=1,
    figsize=figsize,
    pallete=pallete,
    p_correction="fdr_bh",
    annotation_location="outside",
    show=True,
    # truncate_outliers=True
)

# NOTE: R5 has smallest cells in p23-25, which is also highest mito density. Also note plate 3 has lysosome staining abnormalities in the first two cols


In [ ]:
# Lineage Version
order = get_all_group_order()
feature_meas = "AreaShape_Area"  # Mean_Lysosomes_Distance_Centroid_Nuclei_PerCell_Area" #Mean_Lysosomes_DiameterRatio_PerCell"
ylabel = None  # "Mitochondrial Density Per Cell (relative to youngest passage)"#None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"
stat_grouping = "Lineage"

pairs = getpairs(combined_cell_df_mitolyso, group, order)

this_df = extrafeatures_filtered_cell_df_mitolyso.copy()
data_df = this_df

# Uncomment this if the fix to add "doxo_" to the doxo lineages is not set
# no_drug_values = {None,"", "None", "none", "NoDrug", "Control", "control"}

# drug_clean = data_df["Drug"].astype(str).str.strip()
# data_df["Lineage"] = data_df["Lineage"].where(
#     data_df["Drug"].isna() | drug_clean.isin(no_drug_values),
#     drug_clean + "_" + data_df["Lineage"].astype(str),
# )

data_df["Lineage"] = data_df["Lineage"].replace(
    "LIN3-0A-b2-s2-ss1-ss1", "LIN3-0A-b2-s2-ss1-sss1"
)
#     data_df["Lineage"].equals("LIN3-0A-b2-s2-ss1-ss1"),
#     data_df["Lineage"].equals("LIN3-0A-b2-s2-ss1-sss1"),
# )


def get_hard_code_lineage_colours(
    df, lineage_col_name="Lineage", plate_col_name="Plate_Number"
):
    """
    Returns a dictionary mapping each unique lineage number to a hard-coded color.
    This ensures color consistency for each Plate_Number in seaborn/matplotlib plots.
    """
    # Get colour codes from my csv and then convert it into a dict in the {key: #hexcode} format
    colour_codes = pd.read_csv("proliferation_growth_curves/Lineage_only_hexcolour.csv")
    hard_pallete_dict = dict(
        zip(colour_codes[lineage_col_name], colour_codes["Colour"])
    )

    # print(hard_pallete_dict)
    unique_lineages = sorted(df[lineage_col_name].drop_duplicates())
    # display(unique_lineages)

    # If more lineages than colors, fill in missing colours by using a default seaborn color_palette
    for lineage in unique_lineages:
        if lineage not in hard_pallete_dict:
            extra_colours = sns.color_palette("tab20", len(unique_lineages)).as_hex()
            hard_pallete_dict = {
                lineage: extra_colours[i] for i, lineage in enumerate(unique_lineages)
            }
    return hard_pallete_dict


lindict = {
    "LIN1-0B-b1": "#8000FF",
    "LIN2-0A-b1": "#304CC9",
    "LIN3-0A-b2": "#02FD68",
    "LIN4-0B-b2": "#FFF604",
    "LIN5-0B-b3": "#FFDC00",
    "LIN6-0C-b1": "#FFA000",
    "LIN2-0A-b1-s1": "#87A5FF",
    "LIN3-0A-b2-s1": "#01FDBD",
    "LIN4-0B-b2A-s1": "#E0E102",
    "LIN5-0B-b3-s1": "#958300",
    "LIN6-0C-b1-s1": "#FFB479",
    "LIN3-0A-b2-s2-ss1": "#87FE02",
    "LIN7-0C-b3": "#FF7100",
    "LIN8-0B-b2B-s1": "#FF3200",
    "LIN9-0C-b2-s1": "#FF768A",
    "LIN10-0C-b4": "#A770A3",
    "Doxo_LIN13-0C-b2-s3": "#808080",
    "LIN3-0A-b2-s1-ss1": "#00F0FF",
    "LIN3-0A-b2-s1-ss1-sss1": "#00D7FF",
    "LIN3-0A-b2-s1-ss2": "#019BFF",
    "LIN3-0A-b2-s2-ss1-sss1": "#D5FF93",
    "LIN7-0C-b3-s1": "#FF8F85",
    "LIN8-0B-b2B-s1-ss1": "#D2372C",
    "LIN8-0B-b2B-s1-ss2": "#8C1600",
    "LIN6-0C-b1-s2": "#FFD7AD",
    "LIN11-0C-b5": "#FF2E9D",
    "Doxo_LIN11-0C-b5": "#780A78",
    "LIN12-0C-b2-s2": "#FF23FF",
    "Doxo_LIN12-0C-b2-s2": "#BE12D9",
    "LIN2-0A-b1-s1-ss1": "#A8DAFC",
    "LIN7-0C-b3-s2": "#D0A38C",
    "LIN11-0C-b5-s1": "#FFBFFF",
}
# Map each plate to a color and hard code that shit
colour_dict = get_hard_code_lineage_colours(data_df)
pallete = lindict

remove_outliers = False
reps_to_exclude = []
plot_dir = "plots/lineage"
os.makedirs(plot_dir, exist_ok=True)

figsize = (11, 18)

single_feature_super_splitviolinplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    plate_col_name=stat_grouping,
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=True,
    order=order,
    test="tukey_v3",
    reps_to_exclude=reps_to_exclude,
    show_hist=False,  # True,
    remove_outliers=False,
    truncate_outliers=True,
    legend=True,
    context="talk",
    figsize=figsize,
    pallete=pallete,
    shapiro=False,
    # p_correction="fdr_bh",
    # truncate_outliers=True
)

# NOTE: R5 has smallest cells in p23-25, which is also highest mito density


## Make multiple plots as defined

In [ ]:
# take feature : label pairs to be used for plots from csv
features_df = pd.read_csv("CP_features_for_plots.csv")
display(features_df[features_df["feature"] == "AreaShape_Area"])

data_df = extrafeatures_filtered_cell_df_mitolyso.copy()
# Use in plotting loops with guaranteed alignment:


def make_feature_plots_from_csv(
    data_df,
    features_df,
    xlabel="Age Groups",
    group="AllGroups",
    analysis_mode="Plates",
    norm=False,
    order=None,
    figsize=(10, 10),
    truncate_outliers=False,
    annotate_pval=True,
    test="tukey_v3",
    font_scale=1.0,
    annotation_location="inside",
    context="poster",
    show_legend=False,
):
    features_for_plots = features_df.to_dict("records")

    feature_df_cols = [item["feature"] for item in features_for_plots]
    feature_labels = [item["label"] for item in features_for_plots]

    if order is None:
        order = get_all_group_order()

    # Set stats variables and color palette based on analysis mode
    if analysis_mode == "Lineages":
        stat_grouping = "Lineage"
        plot_dir = Path("plots/lineages")
        show_legend = True
        colour_dict = get_hard_code_lineage_colours(data_df)
    elif analysis_mode == "Plates":
        stat_grouping = "Plate_Number"
        colour_dict = get_hard_code_plate_colours(data_df)
        if norm:
            data_df = norm_combined_cell_df_mitolyso.copy()
            feature_labels = [
                f"{name} (normalized to youngest group)" for name in feature_labels
            ]
            plot_dir = Path("plots/norm")
        else:
            data_df.copy()
            plot_dir = Path("plots/notnorm")

    else:
        raise ValueError(
            f"Invalid analysis mode: {analysis_mode}. Use 'Lineages' or 'Plates'."
        )
    pallete = colour_dict  # "pastel"

    if annotate_pval is False:
        test = None
        plot_dir = Path(plot_dir, "simplified")
    os.makedirs(plot_dir, exist_ok=True)
    for i, feature in enumerate(feature_df_cols):
        ylabel = feature_labels[i]

        single_feature_super_boxplot(
            data_df,
            x_value=group,
            y_value=feature,
            plate_col_name=stat_grouping,
            xtitle=xlabel,
            ytitle=ylabel,
            out_dir=plot_dir,
            annotate=annotate_pval,
            order=order,
            test=test,
            truncate_outliers=truncate_outliers,
            legend=show_legend,
            context=context,
            figsize=figsize,
            pallete=pallete,
            p_correction="fdr_bh",
            shapiro=False,
            font_scale=font_scale,
            show=False,
            annotation_location=annotation_location,
        )
        # single_feature_super_splitviolinplot(
        #     data_df,
        #     x_value=group,
        #     y_value=feature,
        #     plate_col_name=stat_grouping,
        #     xtitle=xlabel,
        #     ytitle=ylabel,
        #     out_dir=plot_dir,
        #     annotate=annotate_pval,
        #     order=order,
        #     test=test,
        #     reps_to_exclude=reps_to_exclude,
        #     show_hist=False,  # True,
        #     remove_outliers=remove_outliers,
        #     rm_outliers_method=rm_outliers_method,
        #     truncate_outliers=truncate_outliers,
        #     legend=show_legend,
        #     context=context,
        #     figsize=figsize,
        #     pallete=pallete,
        #     p_correction="fdr_bh",
        #     shapiro=False,
        #     font_scale=font_scale,
        #     show=False,
        #     annotation_location=annotation_location,
        # )


make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=False,
    analysis_mode="Plates",
    truncate_outliers=False,
    figsize=(12, 14),
    annotate_pval=True,
    test="tukey_v3",
    font_scale=1.0,
    annotation_location="outside",
    context="poster",
)

make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=False,
    analysis_mode="Plates",
    truncate_outliers=False,
    figsize=(12, 10),
    annotate_pval=False,
    test=None,
    font_scale=1.0,
    annotation_location="outside",
    context="poster",
)

make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=True,
    analysis_mode="Plates",
    truncate_outliers=False,
    figsize=(12, 14),
    annotate_pval=True,
    test="dunn",
    font_scale=1.0,
    annotation_location="outside",
    context="poster",
)


In [ ]:
def make_a_shitton_of_plots(
    order=[],
    xlabel="Age Groups",
    group="AllGroups",
    pairs=getpairs(combined_cell_df_mitolyso, group, order),
    pallete=colour_dict,
    remove_outliers=False,
    truncate_outliers=True,
    reps_to_exclude=[],
    norm=False,
):
    if not order:
        order = get_all_group_order()

    if norm:
        this_df = norm_combined_cell_df_mitolyso.copy()
        big_out_folder = (
            "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/plots/norm"
        )
    else:
        this_df = extrafeatures_filtered_cell_df_mitolyso.copy()
        big_out_folder = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/plots/notnorm"
    os.makedirs(big_out_folder, exist_ok=True)
    # iterate and make big ball of plots
    for compartment, feature_dict in zip(feature_names, feature_dicts):
        feature_folder = Path(big_out_folder, compartment)
        Path.mkdir(feature_folder, exist_ok=True)
        for feature_type, features in feature_dict.items():
            newfolder = Path(feature_folder, feature_type)
            Path.mkdir(newfolder, exist_ok=True)
            for feature in features:
                single_feature_super_splitviolinplot(
                    this_df,
                    x_value=group,
                    y_value=feature,
                    plate_col_name="Plate_Number",
                    xtitle=xlabel,
                    ytitle=None,
                    out_dir=newfolder,
                    annotate=True,
                    order=order,
                    test="tukey_v3",
                    reps_to_exclude=reps_to_exclude,
                    show_hist=False,
                    remove_outliers=remove_outliers,
                    truncate_outliers=truncate_outliers,
                    show=False,
                    legend=True,
                    context="poster",
                    figsize=(14, 18),
                    pallete=pallete,
                    shapiro=False,
                    p_correction="fdr_bh",
                )


# make_a_shitton_of_plots(reps_to_exclude=[],norm=False)
#make_a_shitton_of_plots(reps_to_exclude=[], norm=True)

## Attempting PCA

In [ ]:
vars = cell_features["areashape"]
vars = vars + nuc_features["areashape"]
vars = vars + mito_features["areashape"]
vars = vars + lyso_features["areashape"]
vars = vars + mito_features["texture"]
vars = vars + lyso_features["texture"]
vars = vars + mito_features["count"]
vars = vars + lyso_features["count"]
# vars = vars + mito_features["radialdistribution"]
vars = vars + mito_features["per_cell_area"]
vars = vars + lyso_features["per_cell_area"]
# var = vars + mito_features["granularity"]


# Stratified sampling by group
def stratified_sample(df, group_col, sample_size, random_state=40):
    """
    Take a stratified sample maintaining group proportions
    """
    # Calculate sampling fraction and sample from each group proportionally
    frac = sample_size / len(df)
    sample_df = df.groupby(group_col, group_keys=False).apply(
        lambda x: x.sample(frac=frac, random_state=random_state), include_groups=True
    )
    return sample_df


sample_df = stratified_sample(this_df, "AllGroups", sample_size=10000)

In [ ]:
from sklearn.decomposition import PCA

n_components = 2
pca = PCA(n_components)

x_df = sample_df[vars.__add__(["AllGroups", "Plate_Number", "AreaShape_Area"])].dropna()
display(x_df)
components = pca.fit_transform(x_df[vars])
display(components)

fig = px.scatter(
    components, x=0, y=1, color=x_df["AllGroups"], symbol=x_df["Plate_Number"]
)
total_var = pca.explained_variance_ratio_.sum() * 100

# labels = {str(i): f"PC {i + 1}" for i in range(n_components)}
# labels["color"] = "AllGroups"

# fig = px.scatter_matrix(
#     components,
#     color=x_df["AllGroups"],
#     dimensions=range(n_components),
#     labels=labels,
#     title=f"Total Explained Variance: {total_var:.2f}%",
# )
# fig.update_traces(diagonal_visible=False)
fig.show()

# labels = {
#     str(i): f"PC {i + 1} ({var:.1f}%)"
#     for i, var in enumerate(pca.explained_variance_ratio_ * 100)
# }

# fig = px.scatter_matrix(
#     components, labels=labels, dimensions=range(4), color=df["species"]
# )
# fig.update_traces(diagonal_visible=False)
# fig.show()


In [ ]:
from umap import UMAP
from sklearn.preprocessing import StandardScaler

scaled_data = StandardScaler().fit_transform(x_df[vars])
reducer = UMAP(random_state=40)
reducer.fit(scaled_data)

embedding = reducer.transform(scaled_data)
# Verify that the result of calling transform is
# idenitical to accessing the embedding_ attribute
assert np.all(embedding == reducer.embedding_)
print(embedding.shape)

import sklearn.cluster as cluster

import sklearn.metrics as metrics


In [ ]:
kmeans_labels = cluster.KMeans(n_clusters=9).fit_predict(scaled_data)

"""
From https://umap-learn.readthedocs.io/en/latest/clustering.html: 
The next thing to be aware of is that when using UMAP for dimension reduction you will want to select different parameters 
than if you were using it for visualization. First of all we will want a larger n_neighbors value 
small values will focus more on very local structure and are more prone to producing fine grained cluster structure 
that may be more a result of patterns of noise in the data than actual clusters. 
In this case well double it from the default 15 up to 30. Second it is beneficial to set min_dist to a very low value. 
Since we actually want to pack points together densely (density is what we want after all) a low value will help,
as well as making cleaner separations between clusters.
In this case we will simply set min_dist to be 0.
"""
clusterable_embedding = UMAP(
    n_neighbors=30,
    min_dist=0.0,
    n_components=2,
    random_state=40,
).fit_transform(scaled_data)

# now plot the embeddings
fig = px.scatter(
    clusterable_embedding,
    x=0,
    y=1,
    color=kmeans_labels.astype(str),  # x_df["Plate_Number"],
    symbol=x_df["AllGroups"].astype(str),
    size=x_df["AreaShape_Area"],
    labels={"color": "kmeans"},
)
fig.show()


# labels = cluster.HDBSCAN(
#     min_samples=10,
#     min_cluster_size=500,
# ).fit_predict(clusterable_embedding)

fig = px.scatter(
    clusterable_embedding,
    x=0,
    y=1,
    color=x_df["Plate_Number"].astype(str),
    # symbol=x_df["Plate_Number"].astype(str),  # x_df["Plate_Number"],
    size=x_df["AreaShape_Area"],
    labels={"color": "Plate Number"},
)
fig.update_legends()
fig.show()


In [ ]:
from umap import plot
# Take a random sample of n rows

new_embedding = UMAP(
    n_components=2,
    n_neighbors=15,
    random_state=40,
).fit(scaled_data)
plot.points(new_embedding, labels=x_df.Plate_Number.astype(str), theme="fire")
plt.savefig("umap_test_2.png")

In [ ]:
# Using one-way anova and Tukey's HSD to compare means of non normalized values
order = get_all_group_order()
feature_meas = "AreaShape_Area"
ylabel = "Area"


group = "AllGroups"
plates = "Plate_Number"
pairs = getpairs(combined_cell_df_mitolyso, group, order)
this_df = extrafeatures_filtered_cell_df_mitolyso.copy()
pallete = "pastel"
remove_outliers = True

feature_df = make_single_feature_df(
    this_df, group=group, feature=feature_meas, plates="Plate_Number"
)
group_avg_df = average_groups_by_plate(
    feature_df, x_value=group, y_value=feature_meas, plates="Plate_Number"
)
group_avg_df_pivot = average_groups_pivot(
    group_avg_df, x_value=group, y_value=feature_meas, plate_col_name="Plate_Number"
)

if remove_outliers is True:
    feature_df = remove_outliers_iqr(feature_df)
    display(feature_df)

display(feature_df)
display(group_avg_df)
display(group_avg_df_pivot)

sns.set_theme(style="ticks")
# sns.set_context("notebook", font_scale=1.9)

plt.figure(figsize=(12, 8))
sns.set_context("talk", font_scale=0.5)
plt.figure(dpi=300)


sns.violinplot(
    data=feature_df,
    x=group,
    y=feature_meas,
    order=order,
    fill=False,
    color="gainsboro",
    cut=1,
    native_scale=True,
    linecolor="k",
    inner=None,
    # inner_kws=dict(box_width = 5)
)

ax = sns.swarmplot(
    data=group_avg_df,
    x=group,
    y=feature_meas,
    hue=plates,
    order=order,
    palette=pallete,
    size=10,
    edgecolor="k",
    linewidth=1,
    dodge=0.5,
)

# use a boxplot to draw the mean line - thinking outside the box :)
sns.boxplot(
    data=group_avg_df,
    x=group,
    y=feature_meas,
    showmeans=True,
    meanline=True,
    meanprops={"color": "dimgray", "ls": "-", "lw": 2.5},
    medianprops={"visible": False},
    whiskerprops={"visible": False},
    zorder=1,
    showfliers=False,
    showbox=False,
    showcaps=False,
    ax=ax,
)

ax.legend_.remove()

sns.despine()
plt.gcf()  # .set_size_inches(10, 6)
plt.xlabel(group)
if ylabel == None:
    plt.ylabel(feature_meas.replace("_", " "))


from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

# Extract the data for each group

# Perform the one-way ANOVA test

# Print the results

pvalues, pairs = anova_with_tukey_posthoc(
    group_avg_df, x_value=group, y_value=feature_meas, display_results=True
)


annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order)
annotator.configure(
    text_format="star", loc="inside", verbose=2, hide_non_significant=True
)
annotator.set_pvalues_and_annotate(pvalues)

plt.savefig("plots/new_plots/" + feature_meas + "_anova_superviolinplot.png", dpi=300)
plt.show()


## Summary Stats

In [ ]:
summary_outpath = "postprocessed_summary_stats"
os.makedirs(summary_outpath, exist_ok=True)

feature_cols = [
    "AreaShape_Area",
    "Nuclei_AreaShape_Area",
    "Cell_Nuclei_Area_Ratio",
    "Nuclei_Intensity_MedianIntensity_DAPI_MAX_WellNormalized",
    "Intensity_MeanIntensity_LAMP1_MAX_WellNormalized",
    "Intensity_MeanIntensity_MitoTracker_MAX_WellNormalized",
    "Neighbors_NumberOfNeighbors_5",
    "Neighbors_PercentTouching_5",
    "Nuclei_Neighbors_NumberOfNeighbors_1",
    "Nuclei_Neighbors_PercentTouching_1",
    "Children_Lysosomes_Count",
    "Children_Mitochondria_Count",
]
include_cols = ["Number_Object_Number"] + feature_cols
print(include_cols)


def make_summary_stats_for_df_and_feature(
    df,
    x_value,
    feature,
    summary_outpath,
    df_tag="original",
    plate_col_name="Plate_Number",
    feature_name="area",
    group_name="passage_group",
    include_cols=[],
    inculded_percentiles=[
        0.01,
        0.025,
        0.05,
        0.1,
        0.25,
        0.5,
        0.75,
        0.9,
        0.95,
        0.975,
        0.99,
    ],
):
    from pathlib import Path

    try:
        table_csvname = f"{df_tag}_total_combined_stats.csv"
        feature_csvname = f"{df_tag}_{feature_name}_by_{group_name}_stats.csv"
        agg_feature_csvname = f"{df_tag}_agg_{feature_name}_by_{group_name}_stats.csv"

        subfolder_name = f"{df_tag}_{feature_name}_summary_stats"
        parent_folder = Path(summary_outpath, subfolder_name)
        parent_folder.mkdir(exist_ok=True)

        if not include_cols:
            df_to_summarize = df
        else:
            df_to_summarize = df[include_cols]
        df_to_summarize.describe(percentiles=inculded_percentiles).to_csv(
            os.path.join(summary_outpath, table_csvname)
        )
        group_averages = df.groupby(
            [x_value, plate_col_name], as_index=False, observed=True
        )[feature]
        # Reset the index to get a clean DataFrame
        # average_df = group_averages.reset_index()
        avg_summary = group_averages.describe(percentiles=inculded_percentiles)
        avg_summary_sorted = avg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        avg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, feature_csvname)
        )

        # do the agg by passage group only
        group_averages_agg = df.groupby([x_value], as_index=False, observed=True)[
            feature
        ]
        avg_agg_summary = group_averages_agg.describe(percentiles=inculded_percentiles)
        avg_agg_summary_sorted = avg_agg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        avg_agg_summary_sorted.T.to_csv(
            os.path.join(summary_outpath, subfolder_name, agg_feature_csvname)
        )
        print(
            f"saved files {(table_csvname, feature_csvname, agg_feature_csvname)} to {summary_outpath}"
        )
        return True
    except ValueError as e:
        print(f"Could not make summary stats: {e}")
        return False


this_df = combined_cell_df_mitolyso #extrafeatures_filtered_cell_df_mitolyso.copy()


for feature in feature_cols:
    df_sorted = this_df.sort_values(
        by=["AllGroups"], key=lambda x: x.map(passage_groups_sort_key)
    ).reset_index(drop=True)
    make_summary_stats_for_df_and_feature(
        df_sorted,
        "AllGroups",
        feature,
        summary_outpath,
        df_tag="original",
        feature_name=feature,
        include_cols=include_cols,
    )
    # seaborn_ridgeplot(
    #     df_sorted,
    #     value_col=feature,
    #     group_col="AllGroups",
    #     palette="Set2",
    #     save=True,
    #     out_dir=summary_outpath,
    #     show_percentiles=True,
    #     truncate_outliers=True,
    # )


### To export the normalized csv:


In [ ]:
combined_cell_df_mitolyso_borders_excluded = pd.read_csv(
    os.path.join(csvpath, filename_borders_excluded)
)
filtered_borders_excluded = apply_all_filters(
    combined_cell_df_mitolyso_borders_excluded
)

for feature in feature_cols:
    make_summary_stats_for_df_and_feature(
        filtered_borders_excluded,
        "AllGroups",
        feature,
        summary_outpath,
        df_tag="borders_excluded",
        feature_name=feature,
        include_cols=include_cols,
    )


In [ ]:
preprocessed_df = extrafeatures_filtered_cell_df_mitolyso.copy()

preprocessed_df.to_csv(
    os.path.join(csvpath, "CellProfiler_features_preprocessed.csv"), index=False
)

norm_cell_df_mitolyso.to_csv(
    os.path.join(csvpath, "Norm_CellProfiler_features_preprocessed.csv"),
    index=False,
)